In [1]:
# Change working directory
from pathlib import Path
import os

path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Spruce")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Spruce


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_Spruce_pre.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Maximum_Rate_CFS": "Pre Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Max Flood Rate (cfs)",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG)
Node,,,
SB5003,0.06,0.75,0.000
sP1894,1.59,7.66,0.060
sP1895,1.56,8.41,0.066
sP1895A,1.53,14.86,0.104
sP1896,1.59,6.50,0.056


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_Spruce_imp1.rpt")

# Extract data
post = post_rpt.node_flooding_summary.copy()

# Ensure Node is index
post.index.name = "Node"

# Rename columns
post = post.rename(columns={
    "Hours_Flooded": "Post Hours Flooded",
    "Maximum_Rate_CFS": "Post Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
})

# Keep only needed columns
post = post[[
    "Post Hours Flooded",
    "Post Max Flood Rate (cfs)",
    "Post Total Flood Vol (MG)"
]]

post.head()

,Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG)
Node,,,
SB5003,0.06,0.75,0.000
sP1897,0.35,13.32,0.031
sP1898,0.36,17.01,0.035
sP1899,0.49,39.35,0.080
sP1899A,0.49,56.41,0.127


In [5]:
# Merge on index (Node)
comparison = pre.join(post, how="outer")

# Fill missing values
fill_cols = [
    "Pre Total Flood Vol (MG)",
    "Post Total Flood Vol (MG)",
    "Pre Hours Flooded",
    "Post Hours Flooded"
]

for col in fill_cols:
    comparison[col] = comparison[col].fillna(0)

# -----------------------------
# Reduction calculations
# -----------------------------

# Flood volume reduction
comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"] - comparison["Post Total Flood Vol (MG)"]
)

comparison["Total Flood Vol Percent Reduction"] = np.where(
    comparison["Pre Total Flood Vol (MG)"] > 0,
    (comparison["Total Flood Vol Reduction (MG)"] / comparison["Pre Total Flood Vol (MG)"]) * 100,
    0
)

# Flood duration reduction
comparison["Hours Flooded Reduction"] = (
    comparison["Pre Hours Flooded"] - comparison["Post Hours Flooded"]
)

comparison["Hours Flooded Percent Reduction"] = np.where(
    comparison["Pre Hours Flooded"] > 0,
    (comparison["Hours Flooded Reduction"] / comparison["Pre Hours Flooded"]) * 100,
    0
)

# Round values
comparison["Total Flood Vol Reduction (MG)"] = comparison["Total Flood Vol Reduction (MG)"].round(3)
comparison["Total Flood Vol Percent Reduction"] = comparison["Total Flood Vol Percent Reduction"].round(1)

comparison["Hours Flooded Reduction"] = comparison["Hours Flooded Reduction"].round(2)
comparison["Hours Flooded Percent Reduction"] = comparison["Hours Flooded Percent Reduction"].round(1)

# Sort by severity
comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)

comparison.head(20)

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG),Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction,Hours Flooded Reduction,Hours Flooded Percent Reduction
Node,,,,,,,,,,
sP1899A,1.08,57.93,0.158,0.49,56.41,0.127,0.031,19.6,0.59,54.6
sP2307A,2.42,9.82,0.125,0.61,48.07,0.056,0.069,55.2,1.81,74.8
sP315C,1.13,39.15,0.117,0.46,39.44,0.080,0.037,31.6,0.67,59.3
sP2305,2.58,9.05,0.116,0.73,36.94,0.074,0.042,36.2,1.85,71.7
sP315B,1.12,33.82,0.109,0.38,23.61,0.048,0.061,56.0,0.74,66.1
sP1895A,1.53,14.86,0.104,0.00,NaN,0.000,0.104,100.0,1.53,100.0
sP315D,0.29,65.76,0.104,0.15,54.67,0.052,0.052,50.0,0.14,48.3
sP1899,1.09,40.69,0.100,0.49,39.35,0.080,0.020,20.0,0.60,55.0
sP315A,1.10,24.42,0.098,0.09,2.07,0.002,0.096,98.0,1.01,91.8


In [6]:
# Reset index so Node becomes a column
export_df = comparison.reset_index()

# Export to CSV
export_df.to_csv("flood_comparison.csv", index=False)

print("Exported: flood_comparison.csv")

Exported: flood_comparison.csv


In [7]:
# -----------------------------
# Entire network flood reduction
# -----------------------------

# --- Number of flooded nodes ---
pre_flooded_nodes = (comparison["Pre Hours Flooded"] > 0).sum()
post_flooded_nodes = (comparison["Post Hours Flooded"] > 0).sum()
flooded_nodes_reduction = pre_flooded_nodes - post_flooded_nodes
flooded_nodes_percent_reduction = float(np.where(
    pre_flooded_nodes > 0,
    (flooded_nodes_reduction / pre_flooded_nodes) * 100,
    0
))

# --- Average flood duration (flooded nodes only) ---
pre_avg_duration = comparison.loc[comparison["Pre Hours Flooded"] > 0, "Pre Hours Flooded"].mean()
post_avg_duration = comparison.loc[comparison["Post Hours Flooded"] > 0, "Post Hours Flooded"].mean()
pre_avg_duration = pre_avg_duration if not pd.isna(pre_avg_duration) else 0.0
post_avg_duration = post_avg_duration if not pd.isna(post_avg_duration) else 0.0
avg_duration_reduction = pre_avg_duration - post_avg_duration
avg_duration_percent_reduction = float(np.where(
    pre_avg_duration > 0,
    (avg_duration_reduction / pre_avg_duration) * 100,
    0
))

# --- Volume ---
network_pre_total = comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total = comparison["Post Total Flood Vol (MG)"].sum()
network_reduction_mg = network_pre_total - network_post_total

network_percent_reduction = np.where(
    network_pre_total > 0,
    (network_reduction_mg / network_pre_total) * 100,
    0
)

# -----------------------------
# Print results
# -----------------------------

print("--- Flood Volume ---")
print(f"Pre-Project Network Total Flood Volume: {network_pre_total:.3f} MG")
print(f"Post-Project Network Total Flood Volume: {network_post_total:.3f} MG")
print(f"Network Flood Volume Reduction: {network_reduction_mg:.3f} MG")
print(f"Network Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

print("\n--- Flooded Nodes ---")
print(f"Pre-Project Number of Flooded Nodes: {pre_flooded_nodes}")
print(f"Post-Project Number of Flooded Nodes: {post_flooded_nodes}")
print(f"Flooded Nodes Reduction: {flooded_nodes_reduction}")
print(f"Flooded Nodes Percent Reduction: {flooded_nodes_percent_reduction:.1f}%")

print("\n--- Average Flood Duration (Flooded Nodes Only) ---")
print(f"Pre-Project Avg Flood Duration: {pre_avg_duration:.2f} hrs")
print(f"Post-Project Avg Flood Duration: {post_avg_duration:.2f} hrs")
print(f"Avg Flood Duration Reduction: {avg_duration_reduction:.2f} hrs")
print(f"Avg Flood Duration Percent Reduction: {avg_duration_percent_reduction:.1f}%")


--- Flood Volume ---
Pre-Project Network Total Flood Volume: 3.923 MG
Post-Project Network Total Flood Volume: 1.771 MG
Network Flood Volume Reduction: 2.152 MG
Network Flood Volume Percent Reduction: 54.9%

--- Flooded Nodes ---
Pre-Project Number of Flooded Nodes: 92
Post-Project Number of Flooded Nodes: 96
Flooded Nodes Reduction: -4
Flooded Nodes Percent Reduction: -4.3%

--- Average Flood Duration (Flooded Nodes Only) ---
Pre-Project Avg Flood Duration: 1.41 hrs
Post-Project Avg Flood Duration: 0.54 hrs
Avg Flood Duration Reduction: 0.87 hrs
Avg Flood Duration Percent Reduction: 62.0%


In [8]:
# Fix: ensure these are scalars (not numpy arrays)
network_percent_reduction = float(network_percent_reduction)
network_hours_percent_reduction = float(network_hours_percent_reduction)

# -----------------------------
# Create summary table
# -----------------------------
summary_df = pd.DataFrame({
    "Description": [
        "Pre-Project Number of Flooded Nodes",
        "Post-Project Number of Flooded Nodes",
        "Flooded Nodes Reduction",
        "Flooded Nodes Percent Reduction (%)",
        "Pre-Project Avg Flood Duration - Flooded Nodes Only (hrs)",
        "Post-Project Avg Flood Duration - Flooded Nodes Only (hrs)",
        "Avg Flood Duration Reduction (hrs)",
        "Avg Flood Duration Percent Reduction (%)",
        "Pre-Project Network Total Flood Volume (MG)",
        "Post-Project Network Total Flood Volume (MG)",
        "Network Flood Volume Reduction (MG)",
        "Network Flood Volume Percent Reduction (%)",
        "Pre-Project Total Hours Flooded (hrs)",
        "Post-Project Total Hours Flooded (hrs)",
        "Total Hours Flooded Reduction (hrs)",
        "Hours Flooded Percent Reduction (%)"
    ],
    "Value": [
        int(pre_flooded_nodes),
        int(post_flooded_nodes),
        int(flooded_nodes_reduction),
        f"{round(flooded_nodes_percent_reduction, 1)}%",
        round(pre_avg_duration, 2),
        round(post_avg_duration, 2),
        round(avg_duration_reduction, 2),
        f"{round(avg_duration_percent_reduction, 1)}%",
        round(network_pre_total, 3),
        round(network_post_total, 3),
        round(network_reduction_mg, 3),
        f"{round(network_percent_reduction, 1)}%",
        round(network_pre_hours, 2),
        round(network_post_hours, 2),
        round(network_hours_reduction, 2),
        f"{round(network_hours_percent_reduction, 1)}%"
    ]
})

# -----------------------------
# Export summary CSV
# -----------------------------
summary_df.to_csv("network_flood_summary.csv", index=False)

print("Exported: network_flood_summary.csv")


NameError: name 'network_hours_percent_reduction' is not defined